# 06 · MLflow Experiment Tracking

Goal:

1. Create a persistent MLflow tracking store in Google Drive.
2. Record the Dummy, Logistic Regression, XGBoost, and PyTorch MLP experiments in one place.
3. Log metrics, parameters, dataset identity, model artifacts, and validation dates.
4. Save Logistic Regression as a reproducible candidate model.
5. Mark the current validation champion without touching the 2026 test set.

This notebook is about **MLOps**, not improving the score.

In [ ]:
!pip install -q mlflow==3.16.0

In [ ]:
from google.colab import drive
from pathlib import Path

import hashlib
import json

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn

from mlflow.tracking import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ai-tech-market-risk")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "metrics"
MODEL_DIR = PROJECT_ROOT / "models"
MLFLOW_DIR = PROJECT_ROOT / "mlflow"

for directory in [
    REPORTS_DIR,
    MODEL_DIR,
    MLFLOW_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

ML_DATA_FILE = PROCESSED_DATA_DIR / "ml_features.csv"
MODEL_METRICS_FILE = REPORTS_DIR / "05_mlp_validation_metrics.csv"

XGB_MODEL_FILE = MODEL_DIR / "xgboost_candidate.json"
XGB_METADATA_FILE = MODEL_DIR / "xgboost_candidate_metadata.json"

MLP_MODEL_FILE = MODEL_DIR / "mlp_candidate.pt"
MLP_PREPROCESSOR_FILE = MODEL_DIR / "mlp_preprocessor.joblib"
MLP_METADATA_FILE = MODEL_DIR / "mlp_candidate_metadata.json"

LOGISTIC_MODEL_FILE = MODEL_DIR / "logistic_regression_candidate.joblib"

TRACKING_DB = MLFLOW_DIR / "mlflow.db"
ARTIFACT_ROOT = MLFLOW_DIR / "artifacts"

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

TRACKING_URI = f"sqlite:///{TRACKING_DB}"
EXPERIMENT_NAME = "ai-tech-market-risk"

print("MLflow:", mlflow.__version__)
print("scikit-learn:", sklearn.__version__)
print("Tracking DB:", TRACKING_DB)
print("Artifact root:", ARTIFACT_ROOT)

## 1. Load the processed dataset and previous validation results

In [ ]:
if not ML_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Missing processed dataset: {ML_DATA_FILE}"
    )

if not MODEL_METRICS_FILE.exists():
    raise FileNotFoundError(
        "Notebook 05 comparison metrics are missing. "
        "Run Notebook 05 first."
    )

ml_data = pd.read_csv(
    ML_DATA_FILE,
    parse_dates=["Date"],
)

ml_data = (
    ml_data
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

model_metrics = pd.read_csv(
    MODEL_METRICS_FILE,
    index_col=0,
)

display(model_metrics.round(4))

required_models = {
    "DummyClassifier",
    "LogisticRegression",
    "XGBoost",
    "PyTorchMLP",
}

if not required_models.issubset(
    set(model_metrics.index)
):
    raise ValueError(
        "Expected model rows are missing from saved metrics."
    )

CHAMPION_MODEL = (
    model_metrics["roc_auc"]
    .idxmax()
)

print("Validation champion:", CHAMPION_MODEL)
print(
    "Champion ROC AUC:",
    round(
        model_metrics.loc[
            CHAMPION_MODEL,
            "roc_auc",
        ],
        4,
    ),
)

## 2. Fingerprint the dataset

A cryptographic hash gives us a compact identity for the exact processed dataset used by these experiments.

If the file changes later, its SHA-256 fingerprint changes too.

In [ ]:
def sha256_file(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


DATASET_SHA256 = sha256_file(
    ML_DATA_FILE
)

print("Dataset SHA-256:")
print(DATASET_SHA256)

## 3. Recreate the same chronological train and validation periods

The 2026 test period is deliberately excluded from all model evaluation in this notebook.

In [ ]:
FEATURE_COLUMNS = [
    "Return_1D",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D",
    "Volume_Change_1D",
    "Relative_Volume_20D",
    "Price_vs_MA_5D",
    "Price_vs_MA_20D",
    "SPY_Return_1D",
    "QQQ_Return_1D",
    "SMH_Return_1D",
    "SPY_Return_5D",
    "QQQ_Return_5D",
    "SMH_Return_5D",
    "Excess_vs_QQQ_1D",
    "Excess_vs_SMH_1D",
]

CATEGORICAL_COLUMNS = ["Ticker"]
MODEL_COLUMNS = FEATURE_COLUMNS + CATEGORICAL_COLUMNS

TARGET_COLUMN = "Large_Move_5D"
TARGET_HORIZON_DAYS = 5

VALIDATION_START = pd.Timestamp("2025-01-01")
TEST_START = pd.Timestamp("2026-01-01")


def purge_last_trading_dates(data, n_dates):
    unique_dates = np.array(
        sorted(data["Date"].unique())
    )

    if len(unique_dates) <= n_dates:
        raise ValueError(
            "Not enough dates to apply purge."
        )

    purged_dates = unique_dates[-n_dates:]

    return (
        data[
            ~data["Date"].isin(purged_dates)
        ].copy(),
        purged_dates,
    )


train_data = ml_data[
    ml_data["Date"] < VALIDATION_START
].copy()

validation_data = ml_data[
    (ml_data["Date"] >= VALIDATION_START)
    & (ml_data["Date"] < TEST_START)
].copy()

train_data, purged_train_dates = purge_last_trading_dates(
    train_data,
    TARGET_HORIZON_DAYS,
)

validation_data, purged_validation_dates = purge_last_trading_dates(
    validation_data,
    TARGET_HORIZON_DAYS,
)

print(
    "Train:",
    train_data["Date"].min().date(),
    "to",
    train_data["Date"].max().date(),
    len(train_data),
    "rows",
)

print(
    "Validation:",
    validation_data["Date"].min().date(),
    "to",
    validation_data["Date"].max().date(),
    len(validation_data),
    "rows",
)

print("2026 model predictions are not evaluated here.")

## 4. Rebuild and verify Logistic Regression

Notebook 03 did not persist the Logistic Regression pipeline. Because it is our current champion, we recreate it exactly, verify its validation metrics against the previously saved values, then save it.

In [ ]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            FEATURE_COLUMNS,
        ),
        (
            "ticker",
            categorical_transformer,
            CATEGORICAL_COLUMNS,
        ),
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            preprocessor,
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                solver="lbfgs",
                random_state=42,
            ),
        ),
    ]
)

X_train = train_data[MODEL_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].astype(int).copy()

X_validation = validation_data[MODEL_COLUMNS].copy()
y_validation = validation_data[TARGET_COLUMN].astype(int).copy()

logistic_pipeline.fit(
    X_train,
    y_train,
)

logistic_predictions = logistic_pipeline.predict(
    X_validation
)

logistic_probabilities = (
    logistic_pipeline
    .predict_proba(X_validation)[:, 1]
)

recomputed_logistic_metrics = {
    "accuracy": accuracy_score(
        y_validation,
        logistic_predictions,
    ),
    "precision": precision_score(
        y_validation,
        logistic_predictions,
        zero_division=0,
    ),
    "recall": recall_score(
        y_validation,
        logistic_predictions,
        zero_division=0,
    ),
    "f1": f1_score(
        y_validation,
        logistic_predictions,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        logistic_probabilities,
    ),
    "pr_auc": average_precision_score(
        y_validation,
        logistic_probabilities,
    ),
}

pd.Series(
    recomputed_logistic_metrics
).round(6)

In [ ]:
saved_logistic_metrics = (
    model_metrics
    .loc[
        "LogisticRegression",
        list(recomputed_logistic_metrics.keys()),
    ]
    .astype(float)
)

for metric_name, metric_value in (
    recomputed_logistic_metrics.items()
):
    if not np.isclose(
        metric_value,
        saved_logistic_metrics[
            metric_name
        ],
        rtol=1e-6,
        atol=1e-6,
    ):
        raise ValueError(
            f"Logistic metric mismatch: {metric_name}"
        )

joblib.dump(
    logistic_pipeline,
    LOGISTIC_MODEL_FILE,
)

reloaded_logistic = joblib.load(
    LOGISTIC_MODEL_FILE
)

reload_probabilities = (
    reloaded_logistic
    .predict_proba(X_validation)[:, 1]
)

if not np.allclose(
    logistic_probabilities,
    reload_probabilities,
    rtol=1e-10,
    atol=1e-12,
):
    raise ValueError(
        "Reloaded Logistic Regression predictions differ."
    )

print("Logistic Regression verification passed.")
print("Saved:", LOGISTIC_MODEL_FILE)

## 5. Configure persistent MLflow storage

MLflow metadata is stored in SQLite.

Artifacts such as models and metadata are stored in the project folder on Google Drive.

In [ ]:
mlflow.set_tracking_uri(
    TRACKING_URI
)

client = MlflowClient()

existing_experiment = (
    client.get_experiment_by_name(
        EXPERIMENT_NAME
    )
)

if existing_experiment is None:
    experiment_id = client.create_experiment(
        EXPERIMENT_NAME,
        artifact_location=(
            ARTIFACT_ROOT.as_uri()
        ),
    )

    print(
        "Created MLflow experiment:",
        experiment_id,
    )

else:
    experiment_id = existing_experiment.experiment_id

    print(
        "Using existing MLflow experiment:",
        experiment_id,
    )

mlflow.set_experiment(
    EXPERIMENT_NAME
)

experiment = client.get_experiment(
    experiment_id
)

print("Experiment:", experiment.name)
print("Artifact location:", experiment.artifact_location)

## 6. Prepare common experiment metadata

In [ ]:
COMMON_PARAMS = {
    "target": TARGET_COLUMN,
    "target_horizon_trading_days": TARGET_HORIZON_DAYS,
    "large_move_threshold": 0.03,
    "numeric_feature_count": len(FEATURE_COLUMNS),
    "ticker_feature": True,
    "train_start": str(
        train_data["Date"].min().date()
    ),
    "train_end": str(
        train_data["Date"].max().date()
    ),
    "validation_start": str(
        validation_data["Date"].min().date()
    ),
    "validation_end": str(
        validation_data["Date"].max().date()
    ),
}

COMMON_TAGS = {
    "project": "ai-tech-market-risk",
    "evaluation": "2025-validation",
    "test_set_evaluated": "false",
    "dataset_sha256": DATASET_SHA256,
}

train_dataset = mlflow.data.from_pandas(
    train_data[
        MODEL_COLUMNS + [TARGET_COLUMN]
    ],
    source=str(ML_DATA_FILE),
    name="market-risk-training-data",
)

validation_dataset = mlflow.data.from_pandas(
    validation_data[
        MODEL_COLUMNS + [TARGET_COLUMN]
    ],
    source=str(ML_DATA_FILE),
    name="market-risk-validation-data",
)

## 7. Helper for logging metrics consistently

In [ ]:
METRIC_COLUMNS = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]


def log_saved_metrics(model_name):
    row = model_metrics.loc[
        model_name,
        METRIC_COLUMNS,
    ].astype(float)

    for metric_name, value in row.items():
        mlflow.log_metric(
            f"validation_{metric_name}",
            float(value),
        )

## 8. Log the Dummy baseline

In [ ]:
with mlflow.start_run(
    experiment_id=experiment_id,
    run_name="dummy-baseline",
) as run:
    mlflow.log_params(COMMON_PARAMS)

    mlflow.log_param(
        "model_type",
        "DummyClassifier",
    )
    mlflow.log_param(
        "strategy",
        "prior",
    )

    mlflow.set_tags(
        {
            **COMMON_TAGS,
            "candidate_role": "baseline",
            "is_validation_champion": (
                str(
                    CHAMPION_MODEL
                    == "DummyClassifier"
                ).lower()
            ),
        }
    )

    mlflow.log_input(
        train_dataset,
        context="training",
    )

    mlflow.log_input(
        validation_dataset,
        context="validation",
    )

    log_saved_metrics(
        "DummyClassifier"
    )

    dummy_run_id = run.info.run_id

print("Dummy run:", dummy_run_id)

## 9. Log Logistic Regression

MLflow stores the complete scikit-learn pipeline, including standardization and one-hot encoding.

An input example is supplied so MLflow can infer the model input signature.

In [ ]:
with mlflow.start_run(
    experiment_id=experiment_id,
    run_name="logistic-regression",
) as run:
    mlflow.log_params(COMMON_PARAMS)

    mlflow.log_params(
        {
            "model_type": "LogisticRegression",
            "solver": "lbfgs",
            "max_iter": 2000,
            "random_state": 42,
            "numeric_scaling": "StandardScaler",
            "ticker_encoding": "OneHotEncoder",
        }
    )

    mlflow.set_tags(
        {
            **COMMON_TAGS,
            "candidate_role": "candidate",
            "is_validation_champion": (
                str(
                    CHAMPION_MODEL
                    == "LogisticRegression"
                ).lower()
            ),
        }
    )

    mlflow.log_input(
        train_dataset,
        context="training",
    )

    mlflow.log_input(
        validation_dataset,
        context="validation",
    )

    log_saved_metrics(
        "LogisticRegression"
    )

    input_example = (
        X_train
        .head(3)
        .copy()
    )

    mlflow.sklearn.log_model(
        sk_model=logistic_pipeline,
        name="model",
        input_example=input_example,
    )

    mlflow.log_artifact(
        str(LOGISTIC_MODEL_FILE),
        artifact_path="candidate_files",
    )

    logistic_run_id = run.info.run_id

print("Logistic run:", logistic_run_id)

## 10. Log XGBoost candidate artifacts

In [ ]:
if not XGB_MODEL_FILE.exists():
    raise FileNotFoundError(
        "XGBoost candidate model is missing. "
        "Run Notebook 04 first."
    )

if not XGB_METADATA_FILE.exists():
    raise FileNotFoundError(
        "XGBoost metadata is missing."
    )

with open(
    XGB_METADATA_FILE,
    "r",
    encoding="utf-8",
) as file:
    xgb_metadata = json.load(file)

with mlflow.start_run(
    experiment_id=experiment_id,
    run_name="xgboost",
) as run:
    mlflow.log_params(COMMON_PARAMS)

    mlflow.log_param(
        "model_type",
        "XGBoost",
    )

    mlflow.log_param(
        "best_n_estimators",
        xgb_metadata[
            "best_n_estimators"
        ],
    )

    for key, value in (
        xgb_metadata[
            "parameters"
        ].items()
    ):
        mlflow.log_param(
            f"xgb_{key}",
            value,
        )

    mlflow.set_tags(
        {
            **COMMON_TAGS,
            "candidate_role": "candidate",
            "is_validation_champion": (
                str(
                    CHAMPION_MODEL
                    == "XGBoost"
                ).lower()
            ),
        }
    )

    mlflow.log_input(
        train_dataset,
        context="training",
    )

    mlflow.log_input(
        validation_dataset,
        context="validation",
    )

    log_saved_metrics(
        "XGBoost"
    )

    mlflow.log_artifact(
        str(XGB_MODEL_FILE),
        artifact_path="candidate_files",
    )

    mlflow.log_artifact(
        str(XGB_METADATA_FILE),
        artifact_path="candidate_files",
    )

    xgb_run_id = run.info.run_id

print("XGBoost run:", xgb_run_id)

## 11. Log PyTorch MLP candidate artifacts

In [ ]:
for required_file in [
    MLP_MODEL_FILE,
    MLP_PREPROCESSOR_FILE,
    MLP_METADATA_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Missing MLP artifact: {required_file}"
        )

with open(
    MLP_METADATA_FILE,
    "r",
    encoding="utf-8",
) as file:
    mlp_metadata = json.load(file)

with mlflow.start_run(
    experiment_id=experiment_id,
    run_name="pytorch-mlp",
) as run:
    mlflow.log_params(COMMON_PARAMS)

    mlflow.log_params(
        {
            "model_type": "PyTorchMLP",
            "best_epoch": (
                mlp_metadata["best_epoch"]
            ),
            "batch_size": (
                mlp_metadata["batch_size"]
            ),
            "learning_rate": (
                mlp_metadata["learning_rate"]
            ),
            "weight_decay": (
                mlp_metadata["weight_decay"]
            ),
            "architecture": str(
                mlp_metadata["architecture"]
            ),
        }
    )

    mlflow.set_tags(
        {
            **COMMON_TAGS,
            "candidate_role": "candidate",
            "is_validation_champion": (
                str(
                    CHAMPION_MODEL
                    == "PyTorchMLP"
                ).lower()
            ),
        }
    )

    mlflow.log_input(
        train_dataset,
        context="training",
    )

    mlflow.log_input(
        validation_dataset,
        context="validation",
    )

    log_saved_metrics(
        "PyTorchMLP"
    )

    mlflow.log_artifact(
        str(MLP_MODEL_FILE),
        artifact_path="candidate_files",
    )

    mlflow.log_artifact(
        str(MLP_PREPROCESSOR_FILE),
        artifact_path="candidate_files",
    )

    mlflow.log_artifact(
        str(MLP_METADATA_FILE),
        artifact_path="candidate_files",
    )

    mlp_run_id = run.info.run_id

print("MLP run:", mlp_run_id)

## 12. Inspect the experiment table

This is the practical value of experiment tracking: all runs can be compared without searching through notebook outputs.

In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ],
    order_by=[
        "metrics.validation_roc_auc DESC"
    ],
)

selected_columns = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.validation_accuracy",
    "metrics.validation_precision",
    "metrics.validation_recall",
    "metrics.validation_f1",
    "metrics.validation_roc_auc",
    "metrics.validation_pr_auc",
    "tags.is_validation_champion",
]

experiment_table = (
    runs[selected_columns]
    .copy()
)

experiment_table

## 13. Verify the champion from MLflow

In [ ]:
best_mlflow_run = (
    runs
    .sort_values(
        "metrics.validation_roc_auc",
        ascending=False,
    )
    .iloc[0]
)

print(
    "MLflow best run:",
    best_mlflow_run[
        "tags.mlflow.runName"
    ],
)

print(
    "MLflow best ROC AUC:",
    best_mlflow_run[
        "metrics.validation_roc_auc"
    ],
)

if (
    best_mlflow_run[
        "tags.mlflow.runName"
    ]
    != "logistic-regression"
):
    raise ValueError(
        "Unexpected validation champion in MLflow."
    )

print("MLflow champion verification passed.")

## 14. Load the Logistic Regression model back from MLflow

A tracked model is only useful if it can be restored and used again.

In [ ]:
logged_logistic = (
    mlflow.sklearn.load_model(
        f"runs:/{logistic_run_id}/model"
    )
)

mlflow_probabilities = (
    logged_logistic
    .predict_proba(X_validation)[:, 1]
)

if not np.allclose(
    logistic_probabilities,
    mlflow_probabilities,
    rtol=1e-8,
    atol=1e-10,
):
    raise ValueError(
        "MLflow-loaded Logistic Regression "
        "predictions differ."
    )

print(
    "MLflow model reload verification passed."
)

## 15. Save a compact experiment summary

In [ ]:
SUMMARY_FILE = (
    REPORTS_DIR
    / "06_mlflow_experiment_summary.csv"
)

experiment_table.to_csv(
    SUMMARY_FILE,
    index=False,
)

print("Saved:", SUMMARY_FILE)
print("Exists:", SUMMARY_FILE.exists())
print("Size:", SUMMARY_FILE.stat().st_size, "bytes")

print()
print("MLflow tracking complete.")
print("Current validation champion: Logistic Regression")
print("2026 test set remains unevaluated.")

# What to send back

Send:

1. The printed MLflow version
2. Dataset SHA-256
3. `experiment_table`
4. The MLflow model reload verification message

Once this works, our experiment tracking layer is complete.

The next fast MVP step is **FastAPI inference**, where we package the champion model behind a `/predict` endpoint.